In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

np.random.seed(67) # for reproducibility

In [ ]:
# Link Colab to your Google Account
# You will need to sign in to Google here!

from google.colab import drive
drive.mount('/content/drive') 

# **Section 1: Importing and Data Cleaning**

Before running the code, please download `movies.csv` from the Google Drive link [**here**](https://drive.google.com/drive/folders/1Izp2hQJ9oJig3TowOYx359Rc1vZT_nbP?usp=sharing) and place the data into your Google Drive.

-----------------------

> **Your role:** You are a data analyst helping a streaming platform understand its movie catalogue.

The streaming platform is reviewing its movie catalogue before deciding what content to promote, acquire, or recommend. We need to understand the catalogue and identify patterns, while being careful not to make claims the data cannot support.

Load in the movies dataset:

In [ ]:
movies = pd.read_csv("movies.csv")

> **Guiding Question 1:** What exactly is in this catalogue, and can we trust the data before using it?

Look at the first few rows using `.head()`. You can specify the number of rows within the brackets!

In [ ]:
movies.head()

In [ ]:
movies.shape # returns (rows, columns)

In [ ]:
# how many cells contain null values?
print(movies.isna().sum())

# are there any duplicated rows?
print(f"Duplicates: {movies.duplicated().sum()}")

Homepage and tagline aren't particularly helpful to us, so we can remove those columns entirely:


In [ ]:
movies[['homepage','tagline']].head(2)

In [ ]:
movies.drop(['homepage','tagline'], axis = 1, inplace = True)

*What should we do with columns with missing data?*

For movie `runtime`, we could consider using the mean/median. We should inspect the distribution first. If runtimes are **highly skewed or contain extreme values**, the **median may be more appropriate**.

In [ ]:
print(movies['runtime'].describe())

Wait... notice anything odd about any of the values above?

In [ ]:
# Activity: Checking out anomalies

print(f"Number of films with a runtime of 0: {_____}")

movies = movies[_____]

print(f"{movies[_____]._____} minutes is now the shortest film in our dataset")

Going back to the `runtime` missing values, we can see that the mean is slightly higher than the median. Also, the maximum value (338 minutes) is much higher than the 75th percentile (118 minutes).

Hence, even without a graph, we can see that the data is right-skewed.

But here's a plot of the data using a histogram (more on visualisation to come later!) to confirm that.

In [ ]:
plt.hist(
    movies["runtime"],
    bins=30
); plt.show()

In this case, it would be better to use the **median** to fill in the missing values for `runtime`.

In [ ]:
movies['runtime'] = movies['runtime'].fillna(movies['runtime'].median())

print(f"Median runtime is {movies['runtime'].median()} minutes!")

With the `genre` columns, we don't know what the missing movie genres are, so assigning a specific genre (e.g. using mode) could introduce false information.

Using "Unknown" preserves those observations while clearly indicating that the genre is unavailable.

In [ ]:
genre_cols = [
    'primary_genre',
    'secondary_genre',
    'tertiary_genre'
]

movies[genre_cols] = movies[genre_cols].fillna('Unknown')

For columns like `keywords`, `production_companies/countries`, `overview` and `spoken_languages`, we could leave the null values as they are, since it's valid not to have any keywords, production companies, or languages.

Here, we replace them with an empty string `''` instead, to allow us to treat all entries as text when performing text-based analysis further down the line.

In [ ]:
text_cols = [
    'keywords',
    'production_companies',
    'production_countries',
    'spoken_languages',
    'overview'
]

movies[text_cols] = movies[text_cols]._____

Let's check for null values again:

In [ ]:
print(_____)

`.info()` is a quick way to see the column names, number of observations and data types in your dataframe.

In [ ]:
print(movies.info())

Date columns like `release_date` should be changed to datetime format instead of a string.

In [ ]:
movies["release_date"] = pd.to_datetime(movies['release_date']); movies['release_date'].info()

## **Exploring the Data**

As the data has generally been cleaned of null values, we can start examining the columns of the dataset more closely!

In [ ]:
print(movies.shape) # remaining rows/cols

In [ ]:
# What is the average/mean vote count for all films?
print(f"Mean vote count: {movies['vote_count'].mean().round(2)}")

# What is the most budget of a film in this dataset?
print(f"Max budget: ${movies["budget"]._____}")

To get an overall summary, an easier way is to use `.describe()`:

In [ ]:
movies.describe()

> **Guiding question 2:** Is the catalogue dominated by short (less than 60 mins), medium (60 to 120 mins), long (120 mins to 180 mins), or very long films (180 mins or longer)?

It is possible to split numerical data into categories if it helps with our analysis.

Here, we want to investigate the length of movies in the dataset. We can create a new column called `length_category` that categorises the movies based on their runtime.

In [ ]:
bins = [0, 60, 120, 180, float('inf')] # FYI: the lower bound for "Short" will be 14 here, as that's the shortest movie length
labels = ['Short', 'Medium', 'Long', 'Very Long']

movies['runtime_category'] = pd.cut(
    movies['runtime'],
    bins=bins,
    labels=labels,
    right=False # upper bound is non-inclusive, e.g. 0 <= runtime < 60
)

movies['runtime_category']._____

In [ ]:
movies['runtime_category']._____(normalize = True) * 100

77% of films are between 60 and 120 minutes long (medium length).

> **Guiding question 3a:** How many genres are there, and how many films of each primary genre are there?

In [ ]:
print(movies['primary_genre'].nunique()) # how many unique categories are there?

print(movies['primary_genre'].unique()) # what are these categories?

In [ ]:
print(movies['primary_genre'].value_counts())

### **Grouping, Sorting & Aggregation**

By grouping the films into their respective categories, we can look at how the groups compare to one another.

> **Guiding question 3b:** Which 5 primary genres have the highest average popularity?

In [ ]:
movies.groupby('primary_genre')['popularity'].mean()

... and then sort this to find the 5 most popular genres!

In [ ]:
# sort films by vote average, from highest to lowest
movies.groupby('primary_genre')['popularity'].mean().sort_values(ascending=_____).head().round(2)

> **Guiding Question 4:** What are the most common release years in the catalogue?

Look more closely at **release_date**:

In [ ]:
movies['release_date'].sample(10)

We could add a `Year` column to make working with the data easier.

In [ ]:
movies['year'] = movies['release_date'].dt.year; movies['year'].head()

In [ ]:
movies.groupby('year')['year'].count().sort_values(ascending=False).head()

The 5 most featured years are 2009, 2006, 2014, 2013, and 2008.

> **Guiding question 5:** How does the runtime of movies vary across genres? What is the standard deviation of runtimes?

In [ ]:
movies.groupby('primary_genre')['runtime'].agg(
    ['count', 'mean', 'std']
).sort_values('mean', ascending=False).round(2)

# **Section 2: Data Visualisation**

In this section, we will learn how to plot a few different visualisations using two different sets of data `movies.csv` and `immigrants.csv`. Do ensure that **both** sets of data are present in your workspace before beginning!

-------

## **Correlation Matrix**


**Best For:** Exploring Relationships Between Two Different Variables

**What to look out for?**
+ Which variables have the strongest relationships?
+ Is the relationship positive or negative?
+ Are the variables highly correlated with each other?

Note that correlation matrices only work on **numeric** variables only, and hence we will need to subset the columns for only numeric variables first before plotting!

If you would like to test the correlation between two **categorical** variables, you can look into the following instead (but it will not be covered in this workshop):
+ Cramér's V
+ Theil's U (Uncertainty Coefficient)
+ Phi Coefficient (for 2x2 categorical variables)

In [ ]:
movies.dtypes # check for which columns contain numeric values

# let's select four numeric columns to observe correlation
numeric_columns = ["______", "______", "______", "______"] 
movies_numeric = movies[numeric_columns]

# calculate correlation matrix
correlation_matrix = movies_numeric.______()

# plot correlation matrix
plt.figure(figsize = (10, 8))

sns.______(
    correlation_matrix,
    cmap = "______",
    annot = True,
    fmt = ".2f",
    linewidths = 0.5
)

plt.title("______")

## **Word Cloud**

This is a **domain-specific** type of plot that works best when we have word information present in the dataset (e.g. reviews, movie descriptions). Here, we will plot a word cloud of the words specified in the `keywords` column!

In [ ]:
# Split each movie's keywords into a list, using '|' as a separator
movies['keywords'] = movies['keywords'].str.split('|')

# Flatten the lists of keywords into one list, and remove any empty keywords
all_keywords = [
    keyword
    for keywords in movies['keywords']
    for keyword in keywords
    if keyword
]

# Join all the keywords into one single string, separated by spaces
# E.g. "love based novel relationship dystopia ..."
combined_text = " ".join(all_keywords)

# Create a word cloud with the combined keyword text
wordcloud = ______(
    width = 1000,
    height = 500,
    background_color="white"
).generate(combined_text)

# Display the word cloud
plt.figure(figsize=(14, 7))
plt.______(wordcloud)
plt.axis("off")
plt.show()

### **Immigrants Dataset**

For the next few plots that we will be drawing, we will be using the `immigrants.csv` dataset. Here is the data dictionary:

|Variable|Description|Data Type|
|--|--|--|
|**Entity**|Name of Country or Territory|`string`|
|**Code**|Three-Letter Country Code (e.g. `AFG` for Afghanistan)|`string`|
|**Year**|Year of Observation|`int`|
|**Share of the population that was born in another country**|The percentage of the population who were born abroad|`float`|
|**Total number of international immigrants**|Estimated total number of international immigrants living in a country that year|`int`|
|**GDP per capita**|Gross domestic product per capita in a given country and year in constant international dollars. Values have been adjusted for inflation and cost of living.|`float`|

In [ ]:
### Let's do some simple exploration of the dataset
immigrants = pd.read_csv("immigrants.csv")
immigrants.______()

In [ ]:
### Rename the column headers to something simpler
immigrants.columns = ["country", "code", "year", "share", "total_immigrants", "gdp_per_capita"]
immigrants.head()

For simplicity, we will reduce our scope of the dataset to focus on immigrant data in **European Countries** only. The countries considered are specified in the `europe` list below.

Of course, feel free to explore the data for countries of other continents of your own time after the workshop!

In [ ]:
##### Let's reduce the scope of our observations to EUROPEAN countries only
europe = [
    "Albania", "Austria", "Belgium", "Bosnia and Herzegovina",
    "Bulgaria", "Croatia", "Czechia", "Denmark", "Estonia",
    "Finland", "France", "Germany", "Greece", "Iceland",
    "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg",
    "Malta", "Moldova", "Montenegro", "Netherlands",
    "North Macedonia", "Norway", "Poland", "Portugal",
    "Romania", "San Marino", "Serbia", "Slovakia", "Slovenia",
    "Spain", "Sweden", "Switzerland", "Ukraine", "United Kingdom"
]

europe_immigrants = immigrants[immigrants["country"].isin(europe)]

europe_immigrants.head()

## **Line Plot**

**Best For:** Showing Trends Over Time

**Variables Needed**
+ Horizontal Axis: Ordered / Time Variable
+ Vertical Axis: Numerical Variable

**What to look out for?**
+ Is there an upward or downward trend?
+ Are there any peaks or dips?
+ Are there any sudden changes or repeating patterns?

In [ ]:
##### Let's begin with a simple line plot first
sns.______(
    data = europe_immigrants,
    x = "______",
    y = "______",
    hue = "______",
)

plt.show()

# What's the issue?

In [ ]:
##### Let's make our plot more interesting instead of trying to look at all European countries.
##### We'll focus on only the top 5 European countries with the highest total immigrants in 2024.

##### The following code extracts the top 5 European countries in 2024
# This subsets for 2024 observations of each country
europe_2024 = europe_immigrants[europe_immigrants["year"] == 2024]

# This sorts the total_immigrants column and extracts the top 5 countries with highest total_immigrants
top_5_europe = europe_2024.sort_values("total_immigrants", ascending = False).head(5)["country"]

# Obtains the immigrant information of the top 5 countries from 2000 - 2024
top_5_europe_immigrants = europe_immigrants[europe_immigrants["country"].isin(top_5_europe)].copy()
top_5_europe_immigrants["total_immigrants_mill"] = top_5_europe_immigrants["total_immigrants"] / 1e6  

# Plot the line graph
sns.lineplot(
    data = top_5_europe_immigrants,
    x = "year",
    y = "total_immigrants_mill",
    hue = "country",
)

# Add nice plot elements
plt.xlabel("______")
plt.ylabel("______")
plt.title("Top 5 European Countries with Highest Immigrant Flow")

plt.show()

# What can we do better?

In [ ]:
##### Let's put the countries in the legend to the side of the graphs
# Plot the graph
ax = sns.lineplot(
    data = top_5_europe_immigrants,
    x = "year",
    y = "total_immigrants_mill",
    hue = "country",
    marker = "o",
    legend = False
)

# Add country names at the end of each line
for country, country_df in top_5_europe_immigrants.groupby("country"):
    last_row = country_df.sort_values("year").iloc[-1]

    plt.text(
        last_row["year"] + 0.5,
        last_row["total_immigrants"],
        country,
        va="center"
    )

# Add nice plot elements
plt.xlabel("Year")
plt.ylabel("Total Immigrants (Millions)")
plt.grid(True, axis = "y", alpha = 0.8, linestyle = "--")
plt.title("Top 5 European Countries with Highest Immigrant Flow", fontweight = "bold")

# Give labels a bit of breathing room on the right
plt.xlim(______)

plt.show()

## **Choropleth Map**

**Best For:** Comparing Geographical Areas

**Variables Needed**
+ Geographical Variable: Region / Location
+ Numerical Variable: Value Associated with Each Region

**What to look out for?**
+ Which areas have higher or lower values?
+ Are there any geographical patterns or clusters?
+ Are there any regional outliers?

In [ ]:
##### Loading in European Countries Boundaries and Merging Data
# Load Shapes of World Boundaries
world = gpd.read_file(
    "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson"
)

# Consider only European countries and their boundaries
europe_base = world[world["CONTINENT"] == "Europe"]

# Merge 2010 Immigration Data onto European Country Boundaries
europe_map = europe_base.merge(
    europe_2024,
    left_on="ADM0_A3",
    right_on="code",
    how="left"
)

##### Annotation for Top 3 European Countries with Highest Immigration
# Countries to Annotate (Top 3)
highlight_countries = ["Germany", "United Kingdom", "France"]
highlight = europe_map[europe_map["NAME"].isin(highlight_countries)].copy()

# Select a point in the country's polygon
highlight["label_point"] = highlight.geometry.representative_point()

# Chooses where the label boxes should appear
# The values are stored in the form (longitude, latitude), and can be modified if necessary
label_positions = {
    "United Kingdom": (-18, 60),
    "France": (-12, 46),
    "Germany": (18, 56)
}

##### Plot the Choropleth Map
fig, ax = plt.subplots(figsize = (12, 8),
                       facecolor = "#fff9e7")

plot = europe_map.______(
    column = "______",
    cmap = "______",
    linewidth = 0.8,
    edgecolor = "black",
    legend = True,
    legend_kwds = {
        "label": "Total Immigrant Population (millions)",
        "shrink": 0.7
    },
    missing_kwds = {
        "color": "______",
        "edgecolor": "black",
    },
    ax=ax
)

# Adds country annotations to the figure
for _, row in highlight.iterrows():
    country = row["NAME"]
    immigrants = row["total_immigrants"]

    # point inside the country
    x = row["label_point"].x
    y = row["label_point"].y

    # where the textbox should go
    box_x, box_y = label_positions[country]

    ax.annotate(
        f"{country}\nTotal immigrants: {int(immigrants):,}",
        xy = (x, y),               
        xytext = (box_x, box_y),   
        textcoords = "data",
        ha = "left",
        va = "center",
        fontsize = 9,
        bbox = dict(
            boxstyle = "round,pad=0.4",
            facecolor = "white",
            edgecolor = "black",
            alpha = 0.9
        ),
        arrowprops = dict(
            arrowstyle = "->",
            color = "lavender",
            lw = 1.2
        )
    )

# Zoom the plot into Europe
ax.set_xlim(-25, 45)
ax.set_ylim(34, 72)

# Set the title
ax.set_title(
    "Total Immigrant Population Across European Countries in 2024",
    fontweight = "bold",
    fontsize = 16
)

# Turn off the x and y-axis
ax.axis("off")

# Adds a caption specifying what the colour "grey" means
plt.figtext(
    0.64, 0.14,
    "*Countries in GREY have no immigrant data.",
    ha = "left",
    fontsize = 8,
    style = "italic"
)

plt.show()

# **Section 3: Working with LLMs**

In this section, we will be working with the Food Delivery dataset `food_delivery.csv` which is likewise available at our Google Drive link [here](https://drive.google.com/drive/folders/1Izp2hQJ9oJig3TowOYx359Rc1vZT_nbP?usp=sharing).

We will learn how to take advantage of LLMs to assist in the EDA process through LLM-Assisted Coding, and good LLM usage principles so that we can make the most out of our visualisations!

------

## **Food Delivery Dataset**

### **Understanding the Dataset**

Before analysing individual variables, let's first inspect the dataset with the methods taught in the first section! As a quick refresher:

+ `df.head()` gives us a quick preview of the observations and columns.
+ `df.info()` shows the number of rows, column names, data types and non-missing values. This helps us identify possible data-quality issues.
+ `df.describe()` provides basic summary statistics for numerical variables.

In [ ]:
df = pd.read_csv("food_delivery.csv")

# Preview the dataset
display(df.head())

# Check columns, data types and missing values
df.info()

# View summary statistics for numerical variables
display(df.describe())

### **Good Prompt Example**

Look over the code again to ensure there are no errors. Review the code itself to make sure it is doing what you want it to do. In this case, there are no errors and the values shown are in line with the dataset.

**Prompt:**

> I want to understand the distribution of `delivery_time_min`. Write pandas code to calculate its mean, median, standard deviation, minimum, maximum, and quartiles. Do not modify the data or calculate the values yourself. Provide the code only. I want even those with different formats to included, such as those values with units. For blank data, ignore and do not include them in the calculation. Include outliers. Explain the code.  

In [ ]:
# Temporarily extract the numerical part of delivery_time_min
delivery_time = pd.to_numeric(
    df['delivery_time_min']
    .astype('string')
    .str.extract(r'([-+]?\d*\.?\d+)', expand=False),
    errors='coerce'
)

# Calculate summary statistics
print("Mean:", delivery_time.mean())
print("Median:", delivery_time.median())
print("Standard Deviation:", delivery_time.std())
print("Minimum:", delivery_time.min())
print("Maximum:", delivery_time.max())
print("Quartiles:")
print(delivery_time.quantile([0.25, 0.50, 0.75]))

### **Scatterplot**

### **Bad Prompt Example**

The code below throws an error due to unclarified parameters / boundaries in the data.

**Prompt:**
> Make a scatterplot of distance and delivery time.

In [ ]:
# Create scatterplot
plt.scatter(df["distance_km"], df["delivery_time_min"])

# Labels and title
plt.xlabel("Distance (km)")
plt.ylabel("Delivery Time (minutes)")
plt.title("Distance vs Delivery Time")

plt.show()

### **Good Prompt Example**

The code given below works immediately as the task, context and parameters are provided clearly. But you should still check the code to make sure there is fulfills your goals.

Prompt:
> I want to investigate the relationship between `distance_km` and `delivery_time_min`. Write Matplotlib code for a scatterplot with `distance_km` on the x-axis and `delivery_time_min` on the y-axis. I want even those with different formats to included, such as those values with units. For blank data, ignore and do not include them in the calculation. Do not modify the data, and explain the code.

In [ ]:
# Temporarily extract the numerical part of each column
distance = pd.to_numeric(
    df['distance_km'].astype('string').str.extract(r'([-+]?\d*\.?\d+)', expand=False),
    errors='coerce'
)

delivery_time = pd.to_numeric(
    df['delivery_time_min'].astype('string').str.extract(r'([-+]?\d*\.?\d+)', expand=False),
    errors='coerce'
)

# Keep only rows where both values are available
valid = distance.notna() & delivery_time.notna()

# Create the scatterplot
plt.scatter(distance[valid], delivery_time[valid])

# Label the axes
plt.xlabel('Distance (km)')
plt.ylabel('Delivery Time (min)')

# Add a title
plt.title('Distance vs Delivery Time')

# Display the plot
plt.show()

### **Bar Graph**

### **Bad Prompt Example**

This error happens because `delivery_time_min` is being stored by `pandas` as a string (`str` or `object`) rather than as a numeric column.

**Prompt:**
> Give me the code to create a bar plot comparing delivery time and traffic.

In [ ]:
# Calculate average delivery time for each traffic level
avg_delivery = df.groupby("traffic_level")["delivery_time_min"].mean()

# Create bar plot
plt.bar(avg_delivery.index, avg_delivery.values)

# Labels and title
plt.xlabel("Traffic Level")
plt.ylabel("Average Delivery Time (minutes)")
plt.title("Average Delivery Time by Traffic Level")

plt.show()

### **Good Prompt Example**

**Prompt:**

> Compare the mean delivery_time_min across traffic_level. Write Matplotlib code for a bar plot ordered Low, Medium, High. Use the latest Matplotlib version. Label the x axis with traffic level and the y axis with delivery time. I want even those with different formats to included, such as high and HIGH being together. For blank data, ignore and do not include them in the calcultion. Do not modify the data, and explain the code.

In [ ]:
# Required order
traffic_order = ['Low', 'Medium', 'High']

# Calculate mean delivery time for each traffic level
# .str.strip() removes accidental spaces
# .str.lower() makes the comparison case-insensitive
mean_delivery = [
    df.loc[
        df['traffic_level'].str.strip().str.lower() == level.lower(),
        'delivery_time_min'
    ].mean()
    for level in traffic_order
]

# Create the bar plot
plt.bar(traffic_order, mean_delivery)

# Label the axes
plt.xlabel('Traffic Level')
plt.ylabel('Delivery Time (min)')

# Add a title
plt.title('Mean Delivery Time by Traffic Level')

# Display the plot
plt.show()

### **Histogram**

### **Bad Prompt Example**

An error is thrown because the `delivery_time_min` contains mixed data types. Some values are numeric, while others may contain units such as 46 **min**. There may also be blank values.

**Prompt:**
> Make a histogram of delivery time.

In [ ]:
# Create histogram
plt.hist(df["delivery_time_min"], bins=20)

# Labels and title
plt.xlabel("Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.title("Distribution of Delivery Time")

plt.show()

### **Good Prompt Example**

**Prompt:**
> I want to examine the distribution of `delivery_time_min`. Write Matplotlib code for a histogram with 20 bins with `delivery_time_min` on the x-axis and density frequency on the y axis. I want even those with different formats to included, such as those values with units. For blank data, ignore and do not include them in the calculation. Include outliers. Do not modify the data, and explain the code.


In [ ]:
# Temporarily extract the numerical part of delivery_time_min
delivery_time = pd.to_numeric(
    df['delivery_time_min']
    .astype('string')
    .str.extract(r'([-+]?\d*\.?\d+)', expand=False),
    errors='coerce'
)

# Remove blank or invalid values only
valid_delivery_time = delivery_time.dropna()

# Create histogram with 20 bins and density frequency
plt.hist(valid_delivery_time, bins=20, density=True)

# Label the axes
plt.xlabel('Delivery Time (min)')
plt.ylabel('Density Frequency')

# Add a title
plt.title('Distribution of Delivery Time')

# Display the histogram
plt.show()

# **Your Turn!**

Now that you've got a feel for exploratory data analysis, it's time to get your hands dirty!

We will be working with a dataset called `olympics.csv`, obtained from Kaggle [here](https://www.kaggle.com/datasets/heesoo37/120-years-of-olympic-history-athletes-and-results).

---

The data dictionary below provides a brief description of each variable in the dataset.

`int` = integer | `str` = string | `float` = number with decimals.


|Variable|Meaning|Data Type|
|--|--|--|
|**ID**|Unique identifier for each athlete|`int`|
|**Name**|Athlete's name|`str`|
|**Sex**|Athlete's sex (Male or Female)|`str`|
|**Age**|Athlete's age|`int`|
|**Height**|Athlete's height (cm)|`float`|
|**Weight**|Athlete's weight (kg)|`float`|
|**Team**|Team name|`str`|
|**NOC**|3-letter National Olympic Committee code|`str`|
|**Games**|Year and season of the Games|`str`|
|**Year**|Year of the Games|`int`|
|**Season**|Season of the Games (Summer or Winter)|`str`|
|**City**|Host City of the Games|`str`|
|**Sport**|Name of sport|`str`|
|**Event**|Name of event|`str`|
|**Medal**|Medal won (Gold, Silver, Bronze, or `NA`)|`str`|

# **Your Task**

Using the techniques covered in this workshop - and with some assistance from any LLM of your choice - explore the dataset and see what you can discover!

---

Here's a simple guide to get you started:
+ **Explore and understand the data**. Look out for suspicious / unusual / missing values, and perform any necessary data cleaning.
+ **Come up with one to two questions of interest** that you would like to investigate using the data. Feel free to draw inspiration from the examples we explored earlier.
+ **Create one to two visualisations** that help you to investigate and answer your question(s) of interest.

---

Remember: it's **OKAY** if your visualisations do not reveal a strong or exciting pattern. Finding little to no evidence of a relationship is still a meaningful result.

For example, suppose your question is:
**"Are taller athletes more likely to win a medal?"**

If your analysis does not reveal a clear relationship between height and medals won, report the finding **AS IS**. Do not manipulate the data or alter your visualisations simple to produce the conclusion you expected.

Most importantly, have fun!!! 😊

If you need guidance or help, feel free to approach any of us: See Tien, Natasha, or Jaye.

In [ ]:
### Helper Code
# This code converts the NOC column to a standardised ISO3 format useful for
# for if you would like to plot chloroplete maps.
# The ISO3 format is stored in a new column called "iso3" in the dataframe.

import country_converter as coco

cc = coco.CountryConverter()

olympics["iso3"] = cc.convert(
    names=olympics["noc"],
    to="ISO3"
)